# Notebook 04v2: Train Prefix Adapter (Bug-Fixed)

Changes from v1:
- `enable_thinking=False` in adapter training
- cosine_loss uses softmax (differentiable) instead of argmax
- Uses v2 hidden states and AE

In [ ]:
import os, sys, subprocess
REPO = 'thoughtcomm'
if os.path.exists(REPO):
    os.chdir(REPO)
    subprocess.run(['git', 'pull', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/AUMEZAK/thoughtcomm.git'], check=True)
    os.chdir(REPO)
    subprocess.run(['pip', 'install', '-e', '.', '-q'], check=True)

In [ ]:
import torch
import os
from configs.config import ThoughtCommConfig
from models.model_utils import load_model_and_tokenizer
from models.autoencoder import SparsityRegularizedAE
from models.prefix_adapter import PrefixAdapter
from pipeline.agreement import AgreementReweighter
from training.train_adapter import train_adapter

from google.colab import drive
drive.mount('/content/drive')

config = ThoughtCommConfig.for_qwen_0_6b()
MODEL_TAG = config.model_name.replace('/', '_')
SAVE_DIR = config.save_dir

In [ ]:
# Load LLM
model, tokenizer = load_model_and_tokenizer(
    config.model_name, dtype=config.torch_dtype
)
print(f'Model loaded: {config.model_name}')

In [ ]:
# Load v2 AE and B matrix
ae_dir = os.path.join(SAVE_DIR, f'{MODEL_TAG}_ae_v2')
ae = SparsityRegularizedAE(config.n_h, config.n_z, config.ae_hidden, config.ae_num_layers)
ae.load_state_dict(torch.load(os.path.join(ae_dir, 'ae.pt'), weights_only=True, map_location='cpu'), strict=False)
ae.eval()

B = torch.load(os.path.join(ae_dir, 'B.pt'), weights_only=True, map_location='cpu')
print(f'AE loaded, B shape: {B.shape}')

In [ ]:
# Load v2 hidden states
math_data = torch.load(
    os.path.join(SAVE_DIR, f'{MODEL_TAG}_math_hidden_v2.pt'), weights_only=False
)
H_train = math_data['H']
metadata = math_data['metadata']
print(f'Training data: {H_train.shape}, {len(metadata)} samples')

In [ ]:
# Create adapter and reweighter
reweighter = AgreementReweighter(B, config)
adapter = PrefixAdapter(config.n_z, config.hidden_size, config.adapter_hidden, config.prefix_length)

print(f'Adapter params: {sum(p.numel() for p in adapter.parameters()):,}')
print(f'Reweighter w: {reweighter.w}')

In [ ]:
# Train adapter (with fixed cosine_loss gradient)
adapter, reweighter, loss_hist = train_adapter(
    model, tokenizer, ae, reweighter, adapter,
    H_train, metadata, config, verbose=True
)

print(f'\nFinal loss: {loss_hist[-1]:.4f}')
print(f'Agreement weights: {reweighter.w.data.tolist()}')

In [ ]:
# Plot training loss
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(loss_hist)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Adapter Training Loss (v2: softmax cosine_loss)')
plt.grid(True)
os.makedirs('results', exist_ok=True)
plt.savefig(f'results/04v2_adapter_loss_{MODEL_TAG}.png', dpi=100)
plt.show()

# Check if loss decreased (cosine component should now contribute)
if len(loss_hist) > 5:
    early = sum(loss_hist[:5]) / 5
    late = sum(loss_hist[-5:]) / 5
    print(f'Early avg loss: {early:.4f}')
    print(f'Late avg loss:  {late:.4f}')
    print(f'Improvement: {(early - late) / early * 100:.1f}%')

In [ ]:
# Save adapter and reweighter
adapter_dir = os.path.join(SAVE_DIR, f'{MODEL_TAG}_adapter_v2')
os.makedirs(adapter_dir, exist_ok=True)

torch.save(adapter.state_dict(), os.path.join(adapter_dir, 'adapter.pt'))
torch.save(reweighter.state_dict(), os.path.join(adapter_dir, 'reweighter.pt'))

import json
summary = {
    'model': config.model_name,
    'adapter_params': sum(p.numel() for p in adapter.parameters()),
    'final_loss': loss_hist[-1],
    'agreement_weights': reweighter.w.data.tolist(),
    'num_epochs': config.adapter_epochs,
}
with open(f'results/04v2_adapter_{MODEL_TAG}.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved: adapter.pt, reweighter.pt')